In [1]:
import pandas as pd
import numpy as np
from typing import Dict, List, Tuple

In [2]:
# Define the 2024-2025 NHL Conference and Division structures
CONFERENCES = {
    'Eastern': [
        'BOS', 'BUF', 'CAR', 'CBJ', 'DET', 'FLA', 'MTL', 'NJD', 
        'NYI', 'NYR', 'OTT', 'PHI', 'PIT', 'TBL', 'TOR', 'WSH'
    ],
    'Western': [
        'ANA', 'CGY', 'CHI', 'COL', 'DAL', 'EDM', 'LAK', 'MIN', 
        'NSH', 'SJS', 'SEA', 'STL', 'UTA', 'VAN', 'VEG', 'WPG'
    ]
}

DIVISIONS = {
    'Atlantic': ['TOR', 'TBL', 'FLA', 'OTT', 'MTL', 'DET', 'BUF', 'BOS'],
    'Metropolitan': ['WSH', 'CAR', 'NJD', 'CBJ', 'NYR', 'NYI', 'PIT', 'PHI'],
    'Central': ['WPG', 'DAL', 'COL', 'MIN', 'STL', 'UTA', 'NSH', 'CHI'],
    'Pacific': ['VEG', 'EDM', 'LAK', 'CGY', 'VAN', 'ANA', 'SEA', 'SJS']
}

In [3]:
gfdist_mean = 0.5
gfdist_std = 0.052836
goals_per_game_leagueavg = 5.477298
home_ice_advantage = 0.26

In [4]:
full_season_schedule = pd.read_csv('NHL_schedule_2025.csv')

In [5]:
# This is the module that randomly draws from the talent distribution for all 32 teams

def assign_team_talent(
    gfdist_mean: float,
    gfdist_std: float,
    goals_per_game_leagueavg: float
    ) -> pd.DataFrame:

    """
    Assigns a random GF% representing true team talent to each NHL team, and calculates average GF and GA per game.

    Arguments:
        gfdist_mean: Mean of the normal distribution for GF%
        gfdist_std: Standard deviation of the normal distribution for GF%
        goals_per_game_leagueavg: Average number of goals per game (GF + GA) in the league

    Returns:
        A Dataframe with Team, GF%, average goals for per game, and average goals against per game
    """

    # Official 3-letter codes for all 32 NHL teams in 2024-25
    teams = [
        'ANA', 'BOS', 'BUF', 'CGY', 'CAR', 'CHI', 'COL', 'CBJ', 'DAL', 'DET', 'EDM', 'FLA', 'LAK', 'MIN', 'MTL', 'NSH',
        'NJD', 'NYI', 'NYR', 'OTT', 'PHI', 'PIT', 'SEA', 'SJS', 'STL', 'TBL', 'TOR', 'UTA', 'VAN', 'VEG', 'WPG', 'WSH'
        ]

    # Assign a randomly chosen GF% to each team, with values restricted to fall between 0 and 1
    gfpct_raw = np.random.normal(loc=gfdist_mean, scale=gfdist_std, size=len(teams))
    gfpct = np.clip(gfpct_raw, 0.0, 1.0)
    gfpct_series = pd.Series(gfpct)

    # Ranking for GF%
    gfpct_rank = gfpct_series.rank(method='min', ascending=False).astype(int)

    # Calculate average goals for and goals against per game for each team, using the league average
    gf_game = goals_per_game_leagueavg * gfpct
    ga_game = goals_per_game_leagueavg * (1-gfpct)

    # Create Dataframe to hold this information (CAN THIS BE ADDED TO BY OTHER MODULES????)
    team_data = pd.DataFrame({
        'Team': teams,
        'GF_percent': gfpct,
        'GF_percent_rank': gfpct_rank,
        'GF_per_game': gf_game,
        'GA_per_game': ga_game
    })

    # Set 'Team' as the index for lookup in other modules
    return team_data.set_index('Team')
    

In [6]:
# Module for simulating a regular season game

def simulate_regular_season_game(
    home_team: 'str',
    away_team: 'str',
    team_averages: pd.DataFrame,
    home_ice_adv: float
) -> Dict:

    """
    Simulates a single regular season game using a Poisson process. Each team's expected GF is adjusted for the opponent's GA.

    Arguments:
        home_team: 3-letter code representing the home team
        away_team: 3-letter code representing the away team
        team_averages: DataFrame with team talent and scoring averages from assign_team_talent
        home_ice_adv: Goal value added to the home team's expected GF

    Returns:
        A dictionary with the game's outcome details.
    """

    # Get baseline averages for each team's scoring
    home_gf = team_averages.loc[home_team, 'GF_per_game']
    home_ga = team_averages.loc[home_team, 'GA_per_game']
    away_gf = team_averages.loc[away_team, 'GF_per_game']
    away_ga = team_averages.loc[away_team, 'GA_per_game']

    # Calculated expected goals (lambda) for each team
    home_gf_lambda = ((home_gf + away_ga) / 2) + home_ice_adv
    away_gf_lambda = (away_gf + home_ga) / 2

    # Simulate score as a Poisson process
    home_score = np.random.poisson(lam=home_gf_lambda)
    away_score = np.random.poisson(lam=away_gf_lambda)

    # Determine the winner and allocate points
    home_pts, away_pts = 0, 0

    if home_score > away_score:
        home_pts, away_pts = 2, 0
    elif away_score > home_score:
        home_pts, away_pts = 0, 2
    else: # Tie after regulation, assign a winner randomly
        if np.random.rand() < 0.5:
            home_pts, away_pts = 2, 1
        else:
            home_pts, away_pts = 1, 2

    return {
        'Home': home_team,
        'Away': away_team,
        'Home_GF': home_score,
        'Away_GF': away_score,
        'Home_Pts': home_pts,
        'Away_Pts': away_pts
    }
    

In [7]:
# Module for simulating a playoff game

def simulate_playoff_game(
    home_team: 'str',
    away_team: 'str',
    team_averages: pd.DataFrame,
    home_ice_adv: float
    ) -> str:

    """
    Simulates a single NHL playoff game (no points/goals tracked, just winner).
    The expected goals rate is adjusted for the opponent's expected GA.

    Returns:
        The 3-letter code of the winning team.
    """

    # Get baseline averages for each team's scoring
    home_gf = team_averages.loc[home_team, 'GF_per_game']
    home_ga = team_averages.loc[home_team, 'GA_per_game']
    away_gf = team_averages.loc[away_team, 'GF_per_game']
    away_ga = team_averages.loc[away_team, 'GA_per_game']

    # Calculated expected goals (lambda) for each team
    home_gf_lambda = ((home_gf + away_ga) / 2) + home_ice_adv
    away_gf_lambda = (away_gf + home_ga) / 2

    # Simulate score as a Poisson process
    home_score = np.random.poisson(lam=home_gf_lambda)
    away_score = np.random.poisson(lam=away_gf_lambda)

    # Determine the winner
    winner = None

    if home_score > away_score:
        winner = home_team
    elif away_score > home_score:
        winner = away_team
    else:
        if np.random.rand() < 0.5:
            winner = home_team
        else:
            winner = away_team

    return winner
    

In [8]:
# Module for simulating a regular season. The schedule from 2024-25 is used.

def simulate_regular_season(
    full_season_schedule: pd.DataFrame,
    team_averages: pd.DataFrame,
    home_ice_adv: float
) -> Tuple[pd.DataFrame, pd.DataFrame]:

    """
    Simulates a full NHL regular season using the 2024-25 schedule.

    Arguments:
        team_averages: DataFrame from module 1.
        home_ice_adv: Goal value added to home team's expected GF.

    Returns:
        A tuple of 2 DataFrames:
        1. Standings: Team, GP, Points, GF, GA
        2. Game log: Record of each game played.
    """

    # Get the list of all teams from team_averages
    teams = team_averages.index.tolist()

    # Initialize a DataFrame for standings
    standings = pd.DataFrame(
        0,
        index = teams,
        columns = ['GP', 'PTS', 'W', 'L', 'OTL', 'GF', 'GA', 'GD']
    )

    game_log = []

    # Simulate all the games by iterating over the schedule DataFrame
    for index, row in full_season_schedule.iterrows():
        home_team = row['HOME']
        away_team = row['VISITOR']
        
        # Skip if team data is missing for some reason
        if home_team not in teams or away_team not in teams:
            continue

        result = simulate_regular_season_game(home_team, away_team, team_averages, home_ice_adv)

        game_log.append(result)

        # Update the standings
        standings.loc[home_team, 'GP'] += 1
        standings.loc[away_team, 'GP'] += 1

        standings.loc[home_team, 'GF'] += result['Home_GF']
        standings.loc[away_team, 'GF'] += result['Away_GF']
        standings.loc[home_team, 'GA'] += result['Away_GF']
        standings.loc[away_team, 'GA'] += result['Home_GF']
        standings.loc[home_team, 'GD'] += (result['Home_GF'] - result['Away_GF'])
        standings.loc[away_team, 'GD'] += (result['Away_GF'] - result['Home_GF'])

        standings.loc[home_team, 'PTS'] += result['Home_Pts']
        standings.loc[away_team, 'PTS'] += result['Away_Pts']

        # Classify wins and losses based on point distribution
        if result['Home_Pts'] == 2 and result['Away_Pts'] == 0:
            standings.loc[home_team, 'W'] += 1
            standings.loc[away_team, 'L'] += 1
        elif result['Home_Pts'] == 0 and result['Away_Pts'] == 2:
            standings.loc[home_team, 'L'] += 1
            standings.loc[away_team, 'W'] += 1
        elif result['Home_Pts'] == 2 and result['Away_Pts'] == 1:
            standings.loc[home_team, 'W'] += 1
            standings.loc[away_team, 'OTL'] += 1
        elif result['Home_Pts'] == 1 and result['Away_Pts'] == 2:
            standings.loc[home_team, 'OTL'] += 1
            standings.loc[away_team, 'W'] += 1

    return standings.sort_values(by='PTS', ascending=False), pd.DataFrame(game_log) # Consider losing the game log when doing the full simulation
    

In [9]:
# Function for resolving ties

def resolve_tie(tied_df: pd.DataFrame) -> str:
    """
    Final tiebreaker: if teams are tied on PTS and W, a random choice is used.
    """
    # Use numpy.random.choice for a truly random selection
    winner = np.random.choice(tied_df.index.tolist(), size=1)[0]
    return winner


In [10]:
# Function for finding winner from a group (division, conference, league)

def find_group_winner(standings_df: pd.DataFrame, team_list: List[str]) -> str:
    """
    Finds the winner within a specified group (League, Conference, or Division) 
    using the tie-breaking rules (PTS > W > Random).
    """
    
    # Filter standings for the relevant group
    group_standings = standings_df.loc[standings_df.index.intersection(team_list)]
    
    if group_standings.empty:
        return "N/A"

    # Sort by primary tiebreaker: PTS (highest first)
    top_pts = group_standings['PTS'].max()
    pts_tied_teams = group_standings[group_standings['PTS'] == top_pts]

    if len(pts_tied_teams) == 1:
        return pts_tied_teams.index[0]
        
    # Sort by secondary tiebreaker: W (Wins)
    top_w = pts_tied_teams['W'].max()
    w_tied_teams = pts_tied_teams[pts_tied_teams['W'] == top_w]

    if len(w_tied_teams) == 1:
        return w_tied_teams.index[0]
    
    # Tertiary tiebreaker: Random choice
    return resolve_tie(w_tied_teams)

In [11]:
def get_playoff_teams(standings: pd.DataFrame) -> Dict[str, List[str]]:
    """
    Determines the 16 playoff teams based on division leaders and wild cards.
    """
    
    playoff_teams = {'Eastern': [], 'Western': []}
    
    # 1. Identify the Top 3 in each division (12 teams)
    division_standings = {}
    division_qualifiers = set()
    
    for division_name, team_list in DIVISIONS.items():
        # Get standings for the division and sort by PTS then W
        div_standings = standings.loc[standings.index.intersection(team_list)]
        
        # Sort using the full tiebreaker order for internal ranking consistency: PTS, then W
        div_standings = div_standings.sort_values(by=['PTS', 'W'], ascending=False)
        division_standings[division_name] = div_standings
        
        # Select the top 3 teams
        top_3 = div_standings.head(3).index.tolist()
        division_qualifiers.update(top_3)
        
        # Add to playoff list by conference
        conf = 'Eastern' if division_name in ['Atlantic', 'Metropolitan'] else 'Western'
        playoff_teams[conf].extend(top_3)
        
    # 2. Determine Wild Card Teams (2 per conference)
    
    for conference_name in CONFERENCES.keys():
        all_conf_teams = CONFERENCES[conference_name]
        
        # Teams that did NOT qualify as a Top 3 team
        non_qualifiers = standings.loc[
            (standings.index.isin(all_conf_teams)) & 
            (~standings.index.isin(division_qualifiers))
        ]
        
        # Sort the remaining teams by PTS, then W, to find the Wild Card teams
        wild_card_pool = non_qualifiers.sort_values(by=['PTS', 'W'], ascending=False)

        #if not wild_card_pool.empty:
        #    ranked_wild_card_pool = rank_group_teams(standings, wild_card_pool.index.tolist())
        #    wild_cards = ranked_wild_card_pool[:2]
        #    playoff_teams[conference_name].extend(wild_cards)
            
        # Select the top 2 teams from the wild card pool
        wild_cards = []
        if len(wild_card_pool) >= 2:
            
            # --- Robust Wild Card Selection with Tiebreaker ---
            
            # 1. Get the team with the highest remaining PTS
            wc1_pts = wild_card_pool.iloc[0]['PTS']
            
            # Teams tied for the first wild card spot
            wc1_tied_pool = wild_card_pool[wild_card_pool['PTS'] == wc1_pts]
            
            # Handle tie for WC1 (PTS > W > Random)
            wc1_winner = find_group_winner(wc1_tied_pool, wc1_tied_pool.index.tolist())
            wild_cards.append(wc1_winner)
            
            # Remove the first winner from the pool to find the second
            wc2_pool = wild_card_pool.drop(wc1_winner)
            
            if not wc2_pool.empty:
                # 2. Get the team with the highest remaining PTS for WC2
                wc2_pts = wc2_pool.iloc[0]['PTS']
                
                # Teams tied for the second wild card spot
                wc2_tied_pool = wc2_pool[wc2_pool['PTS'] == wc2_pts]

                # Handle tie for WC2 (PTS > W > Random)
                wc2_winner = find_group_winner(wc2_tied_pool, wc2_tied_pool.index.tolist())
                wild_cards.append(wc2_winner)

            # Add the selected wild cards to the conference list
            playoff_teams[conference_name].extend(wild_cards)
        
        # Ensure only the required 8 teams are kept (3+3+2)
        playoff_teams[conference_name] = playoff_teams[conference_name][:8]

    return playoff_teams


In [12]:
def determine_champions_and_playoffs(
    standings_tuple: Tuple[pd.DataFrame, pd.DataFrame]
    ) -> Dict:
    """
    Identifies the winners of major regular season titles and the 16 playoff teams.
    
    Args:
        standings_tuple: The (standings, game_log) tuple returned from 
                         simulate_regular_season.
                         
    Returns:
        A dictionary containing all title winners and the list of playoff teams.
    """
    
    # Extract the standings DataFrame (the first element of the tuple)
    standings = standings_tuple[0]
    
    # 1. Determine Champions (President's, Conference, Division)
    champions = {
        "Presidents_Trophy_Winner": find_group_winner(standings, standings.index.tolist()),
        "Eastern_Conference_Winner": find_group_winner(standings, CONFERENCES['Eastern']),
        "Western_Conference_Winner": find_group_winner(standings, CONFERENCES['Western']),
        "Atlantic_Division_Winner": find_group_winner(standings, DIVISIONS['Atlantic']),
        "Metropolitan_Division_Winner": find_group_winner(standings, DIVISIONS['Metropolitan']),
        "Central_Division_Winner": find_group_winner(standings, DIVISIONS['Central']),
        "Pacific_Division_Winner": find_group_winner(standings, DIVISIONS['Pacific']),
    }
    
    # 2. Determine Playoff Teams
    playoff_teams = get_playoff_teams(standings)

    # Combine results
    results = {
        "Champions": champions,
        "Playoff_Teams": playoff_teams
    }
    
    return results


In [13]:
def rank_group_teams(standings: pd.DataFrame, team_list: List[str]) -> List[str]:
    
    """
    Ranks a list of teams based on PTS > W > Random, and returns the list of ranked team codes.
    This is used to consistently rank divisional teams and wild card teams.
    """
    
    group_standings = standings.loc[standings.index.intersection(team_list)].copy()
    
    if group_standings.empty:
        return []
        
    # Sort by PTS, then W (pre-sort to minimize random tiebreaks)
    sorted_df = group_standings.sort_values(by=['PTS', 'W'], ascending=False)
    
    final_ranked_teams = []
    #current_idx = 0
    teams_to_process = sorted_df.index.tolist()
    
    while teams_to_process:
        current_team = teams_to_process[0]
        current_pts = sorted_df.loc[current_team, 'PTS']
        current_w = sorted_df.loc[current_team, 'W']
        
        # Identify all teams still tied with the top team that haven't been ranked yet
        tied_block_teams = [
            team for team in teams_to_process
            if sorted_df.loc[team, 'PTS'] == current_pts and sorted_df.loc[team, 'W'] == current_w
        ]

        if len(tied_block_teams) > 1:
            # Resolve the tie randomly among the teams in this block
            tied_sub_df = sorted_df.loc[tied_block_teams]
            winner_of_tie = resolve_tie(tied_sub_df)
            final_ranked_teams.append(winner_of_tie)
            # Remove the winner from the current view of group_standings so it isn't processed again
            teams_to_process.remove(winner_of_tie)
        else:
            # If no teams or only one team are left in the tied block, just add the top one
            final_ranked_teams.append(current_team)
            teams_to_process.remove(current_team)
        
    return final_ranked_teams


In [14]:
def get_division_teams_by_rank(standings: pd.DataFrame) -> Dict[str, List[str]]:
    
    """
    Returns the full ranked list of teams for each of the four divisions (1st to 8th).
    """
    
    division_ranks = {}
    
    for name, teams in DIVISIONS.items():
        # Get the full 8-team ranking for each division
        division_ranks[name] = rank_group_teams(standings, teams)
    
    return division_ranks
    

In [15]:
def get_wild_card_teams_by_rank(standings: pd.DataFrame, playoff_teams: Dict) -> Dict[str, List[str]]:
    
    """
    Ranks the 2 wild card teams within each conference (WC1 vs WC2) based on PTS/W/Random.
    WC1 is the higher seed.
    """
    
    wild_card_ranks = {}
    
    for conf, teams in playoff_teams.items():
        # The playoff_teams list is assumed to contain 6 divisional teams + 2 wild cards.
        # The wild cards are the last two in the list.
        
        wc_teams = teams[-2:]
        
        if len(wc_teams) == 2:
            # Rank the two wild cards to determine WC1 (higher seed) and WC2 (lower seed)
            ranked_wc = rank_group_teams(standings, wc_teams)
            wild_card_ranks[conf] = ranked_wc
        
        else:
            wild_card_ranks[conf] = [] 
            
    return wild_card_ranks


In [16]:
def setup_first_round_matchups(
    standings_tuple: Tuple[pd.DataFrame, pd.DataFrame], 
    playoff_teams_dict: Dict[str, List[str]]
    ) -> Dict[str, Dict]:
    
    """
    Sets up the 8 first-round playoff series with pairings and home-ice advantage 
    based on the NHL's current format.

    Args:
        standings_tuple: The (standings, game_log) tuple from simulate_regular_season.
        playoff_teams_dict: The output dictionary from determine_champions_and_playoffs 
                            containing the 'Playoff_Teams' list.

    Returns:
        A dictionary where keys identify the series and values are dictionaries 
        containing 'Higher_Seed' (with home-ice) and 'Lower_Seed'.
    """
    
    standings = standings_tuple[0]
    playoff_teams = playoff_teams_dict['Playoff_Teams']
    
    # Get all division teams ranked 1-8
    division_ranks = get_division_teams_by_rank(standings)
    
    # Get the two wild card teams in each conference, ranked WC1 (Higher) and WC2 (Lower)
    wc_ranks = get_wild_card_teams_by_rank(standings, playoff_teams)
    
    final_matchups = {}
    
    # --- EASTERN CONFERENCE MATCHUPS ---
    
    # Division 1st Place teams
    atl1 = division_ranks['Atlantic'][0]
    met1 = division_ranks['Metropolitan'][0]
    
    # Wild Card teams (WC1 = higher seed, WC2 = lower seed)
    ec_wc1 = wc_ranks['Eastern'][0] if wc_ranks['Eastern'] else None
    ec_wc2 = wc_ranks['Eastern'][1] if len(wc_ranks['Eastern']) > 1 else None

    if ec_wc1 is None or ec_wc2 is None:
        raise ValueError('Not enough Eastern Conference wild card teams found.')
    
    # Determine the seeding of the Division Winners (DW1 vs DW2) for Wild Card pairing
    #ec_division_winners = [atl1, met1]
    #ec_div_winner_ranks = rank_group_teams(standings, ec_division_winners)
    ecdw_1_seed = find_group_winner(standings, [atl1, met1])
    ecdw_2_seed = atl1 if ecdw_1_seed == met1 else met1
    
    #higher_div_winner = ec_div_winner_ranks[0] # DW1 (higher rank)
    #lower_div_winner = ec_div_winner_ranks[1]  # DW2 (lower rank)
    
    # Wild Card Pairing Rule:
    # DW1 (Higher Div Winner) plays WC2 (Lower Wild Card) -> Home ice goes to DW1
    # DW2 (Lower Div Winner) plays WC1 (Higher Wild Card) -> Home ice goes to DW2
    
    # 1. Divisional Matchups (2nd vs 3rd)
    final_matchups['EC_ATL2'] = {
        'Higher_Seed': division_ranks['Atlantic'][1], # Atlantic 2nd (Home Ice)
        'Lower_Seed': division_ranks['Atlantic'][2]   # Atlantic 3rd
        }
    
    final_matchups['EC_MET2'] = {
        'Higher_Seed': division_ranks['Metropolitan'][1], # Metropolitan 2nd (Home Ice)
        'Lower_Seed': division_ranks['Metropolitan'][2]   # Metropolitan 3rd
        }
    
    # 2. Wild Card Matchups (Div Winner vs Wild Card)
    # Series Name EC_ATL1 or EC_MET1 is determined by which division winner is playing.
    
    # DW1 (Higher) vs WC2 (Lower)
    series_name_ecdw_1 = 'EC_ATL1' if ecdw_1_seed in DIVISIONS['Atlantic'] else 'EC_MET1'
    final_matchups[series_name_ecdw_1] = {'Higher_Seed': ecdw_1_seed, 'Lower_Seed': ec_wc2}
    
    # DW2 (Lower) vs WC1 (Higher)
    series_name_ecdw_2 = 'EC_ATL1' if ecdw_2_seed in DIVISIONS['Atlantic'] else 'EC_MET1'
    final_matchups[series_name_ecdw_2] = {'Higher_Seed': ecdw_2_seed, 'Lower_Seed': ec_wc1}
        
    # --- WESTERN CONFERENCE MATCHUPS ---
    
    # Division 1st Place teams
    cen1 = division_ranks['Central'][0]
    pac1 = division_ranks['Pacific'][0]

    # Wild Card teams
    wc_wc1 = wc_ranks['Western'][0] if wc_ranks['Western'] else None
    wc_wc2 = wc_ranks['Western'][1] if len(wc_ranks['Western']) > 1 else None

    if wc_wc1 is None or wc_wc2 is None:
        raise ValueError('Not enough Western Conference wild card teams.')
    
    # Determine the seeding of the Division Winners (DW1 vs DW2) for Wild Card pairing
    wc_division_winners = [cen1, pac1]
    wc_div_winner_ranks = rank_group_teams(standings, wc_division_winners)
    
    #higher_div_winner_wc = wc_div_winner_ranks[0] # DW1 (higher rank)
    #lower_div_winner_wc = wc_div_winner_ranks[1]  # DW2 (lower rank)

    wcdw_1_seed = find_group_winner(standings, [cen1, pac1])
    wcdw_2_seed = cen1 if wcdw_1_seed == pac1 else pac1
    
    # Wild Card Pairing Rule:
    # DW1 (Higher Div Winner) plays WC2 (Lower Wild Card) -> Home ice goes to DW1
    # DW2 (Lower Div Winner) plays WC1 (Higher Wild Card) -> Home ice goes to DW2

    # 3. Divisional Matchups (2nd vs 3rd)
    
    final_matchups['WC_CEN2'] = {
        'Higher_Seed': division_ranks['Central'][1], # Central 2nd (Home Ice)
        'Lower_Seed': division_ranks['Central'][2]   # Central 3rd
        }
    
    final_matchups['WC_PAC2'] = {
        'Higher_Seed': division_ranks['Pacific'][1], # Pacific 2nd (Home Ice)
        'Lower_Seed': division_ranks['Pacific'][2]   # Pacific 3rd
        }
    
    # 4. Wild Card Matchups (Div Winner vs Wild Card)
    
    # DW1 (Higher) vs WC2 (Lower)
    series_name_wcdw_1 = 'WC_CEN1' if wcdw_1_seed in DIVISIONS['Central'] else 'WC_PAC1'
    final_matchups[series_name_wcdw_1] = {'Higher_Seed': wcdw_1_seed, 'Lower_Seed': wc_wc2}
    
    # DW2 (Lower) vs WC1 (Higher)
    series_name_wcdw_2 = 'WC_CEN1' if wcdw_2_seed in DIVISIONS['Central'] else 'WC_PAC1'
    final_matchups[series_name_wcdw_2] = {'Higher_Seed': wcdw_2_seed, 'Lower_Seed': wc_wc1}
    
    return final_matchups


In [17]:
def simulate_playoff_series(
    team_a_code: str, team_b_code: str, 
    team_averages: pd.DataFrame, 
    home_ice_advantage: float
    ) -> Tuple[str, str]:
    
    """
    Simulates a best-of-seven playoff series, tracking only wins until one team 
    reaches 4.
    
    Team A is assumed to be the higher seed (home ice advantage in games 1, 2, 5, 7).

    Args:
        team_a_code (str): Code for the higher-seeded team (gets home ice 1, 2, 5, 7).
        team_b_code (str): Code for the lower-seeded team (gets home ice 3, 4, 6).
        gf_a, ga_a (float): GF/GA metrics for Team A.
        gf_b, ga_b (float): GF/GA metrics for Team B.
        home_ice_advantage (float): Goal value added to the home team's expected GF.
        
    Returns:
        Tuple[str, str]: (Winner's team code, Loser's team code)
    """
    
    wins = {team_a_code: 0, team_b_code: 0}
    
    # Game numbers where Team A (Higher Seed) is HOME (2-2-1-1-1 format)
    home_games_a = [1, 2, 5, 7] 
    
    # Loop up to a maximum of 7 games
    for game_num in range(1, 8):
        
        # 1. Determine Home and Away teams and their metrics based on game number
        if game_num in home_games_a:
            home_team = team_a_code
            away_team = team_b_code
        else:
            home_team = team_b_code
            away_team = team_a_code

        # 2. Simulate the Game and get the winner code
        game_winner = simulate_playoff_game(
            home_team=home_team, 
            away_team=away_team, 
            team_averages=team_averages,
            home_ice_adv=home_ice_advantage
        )
        
        # 3. Update Wins
        wins[game_winner] += 1
        
        # 4. Check if the series is over
        if wins[team_a_code] == 4:
            return team_a_code, team_b_code # Winner, Loser
        if wins[team_b_code] == 4:
            return team_b_code, team_a_code # Winner, Loser
            
    # Should not be reached, but necessary for function typing
    return "Error", "Error" 
    

In [18]:
def simulate_first_round(
    first_round_matchups: Dict[str, Dict[str, str]],
    team_averages: pd.DataFrame,
    home_ice_advantage: float
    ) -> Dict[str, str]:
    
    """
    Simulates all 8 first-round playoff series.

    Args:
        first_round_matchups: Output from setup_playoff_bracket, 
                              e.g., {'EC_ATL1': {'Higher_Seed': 'CAR', 'Lower_Seed': 'PIT'}, ...}
        team_averages: DataFrame containing 'avg_GF_per_game' and 'avg_GA_per_game'.
        home_ice_advantage: Goal value added to the home team's expected GF.

    Returns:
        A dictionary mapping series name to the winner's 3-letter code.
        e.g., {'EC_ATL1': 'CAR', 'EC_MET2': 'NYR', ...}
    """
    
    round_winners = {}
    
    # Iterate through all 8 series
    for series_name, teams in first_round_matchups.items():
        
        team_a_code = teams['Higher_Seed']
        team_b_code = teams['Lower_Seed']
        
        # Check for missing team data
        if team_a_code not in team_averages.index or team_b_code not in team_averages.index:
            print(f"Warning: Skipping series {series_name} due to missing data for {team_a_code} or {team_b_code}")
            continue

        # Simulate the series
        winner, loser = simulate_playoff_series(
            team_a_code=team_a_code, 
            team_b_code=team_b_code, 
            team_averages=team_averages,
            home_ice_advantage=home_ice_advantage
        )
        
        # Store the winner
        round_winners[series_name] = winner
        
        # Optional: Log the result for inspection
        # print(f"Series {series_name}: {team_a_code} vs {team_b_code}. Winner: {winner}")
        
    return round_winners


In [19]:
def setup_later_round_matchups(
    winners: Dict[str, str], 
    round_name: str, 
    standings: pd.DataFrame
    ) -> Dict[str, Dict[str, str]]:
    
    """
    Sets up matchups for Round 2, Round 3, or the SCF based on fixed brackets and regular season point totals for home-ice advantage.
    """
    
    new_matchups = {}
    
    # --- Round 2 Logic ---
    if round_name == 'Round2':
        # EC_ATL1 Winner vs EC_ATL2 Winner -> EC_ATLF
        team_a = winners['EC_ATL1']
        team_b = winners['EC_ATL2']
        winner_higher_seed = find_group_winner(standings, [team_a, team_b])
        loser_lower_seed = team_a if team_b == winner_higher_seed else team_b
        new_matchups['EC_ATLF'] = {'Higher_Seed': winner_higher_seed, 'Lower_Seed': loser_lower_seed}

        # EC_MET1 Winner vs EC_MET2 Winner -> EC_METF
        team_a = winners['EC_MET1']
        team_b = winners['EC_MET2']
        winner_higher_seed = find_group_winner(standings, [team_a, team_b])
        loser_lower_seed = team_a if team_b == winner_higher_seed else team_b
        new_matchups['EC_METF'] = {'Higher_Seed': winner_higher_seed, 'Lower_Seed': loser_lower_seed}

        # WC_CEN1 Winner vs WC_CEN2 Winner -> WC_CENF
        team_a = winners['WC_CEN1']
        team_b = winners['WC_CEN2']
        winner_higher_seed = find_group_winner(standings, [team_a, team_b])
        loser_lower_seed = team_a if team_b == winner_higher_seed else team_b
        new_matchups['WC_CENF'] = {'Higher_Seed': winner_higher_seed, 'Lower_Seed': loser_lower_seed}

        # WC_PAC1 Winner vs WC_PAC2 Winner -> WC_PACF
        team_a = winners['WC_PAC1']
        team_b = winners['WC_PAC2']
        winner_higher_seed = find_group_winner(standings, [team_a, team_b])
        loser_lower_seed = team_a if team_b == winner_higher_seed else team_b
        new_matchups['WC_PACF'] = {'Higher_Seed': winner_higher_seed, 'Lower_Seed': loser_lower_seed}

    # --- Round 3 Logic (Conference Finals) ---
    elif round_name == 'Round3':
        # ECF: EC_ATLF Winner vs EC_METF Winner
        team_a = winners['EC_ATLF']
        team_b = winners['EC_METF']
        winner_higher_seed = find_group_winner(standings, [team_a, team_b])
        loser_lower_seed = team_a if team_b == winner_higher_seed else team_b
        new_matchups['ECF'] = {'Higher_Seed': winner_higher_seed, 'Lower_Seed': loser_lower_seed}

        # WCF: WC_CENF Winner vs WC_PACF Winner
        team_a = winners['WC_CENF']
        team_b = winners['WC_PACF']
        winner_higher_seed = find_group_winner(standings, [team_a, team_b])
        loser_lower_seed = team_a if team_b == winner_higher_seed else team_b
        new_matchups['WCF'] = {'Higher_Seed': winner_higher_seed, 'Lower_Seed': loser_lower_seed}
    
    # --- Round 4 Logic (SCF) ---
    elif round_name == 'SCF':
        # SCF: ECF Winner vs WCF Winner
        team_a = winners['ECF']
        team_b = winners['WCF']
        winner_higher_seed = find_group_winner(standings, [team_a, team_b])
        loser_lower_seed = team_a if team_b == winner_higher_seed else team_b
        new_matchups['SCF'] = {'Higher_Seed': winner_higher_seed, 'Lower_Seed': loser_lower_seed}

    return new_matchups


In [20]:
def simulate_playoff_round(
    matchups: Dict[str, Dict[str, str]],
    team_averages: pd.DataFrame,
    home_ice_advantage: float
    ) -> Dict[str, str]:
    
    """
    Simulates all series in a given round.
    """
    
    round_winners = {}
    
    for series_name, teams in matchups.items():
        team_a_code = teams['Higher_Seed']
        team_b_code = teams['Lower_Seed']

        # Simulate the series
        winner, _ = simulate_playoff_series(
            team_a_code=team_a_code, 
            team_b_code=team_b_code, 
            team_averages=team_averages,
            home_ice_advantage=home_ice_advantage
            )
        
        round_winners[series_name] = winner
        
    return round_winners

In [21]:
def simulate_full_postseason(
    standings_tuple: Tuple[pd.DataFrame, pd.DataFrame],
    team_averages: pd.DataFrame,
    home_ice_advantage: float
    ) -> Dict[str, str]:
    
    """
    Simulates the entire four-round NHL postseason.

    Args:
        standings_tuple: The output from simulate_regular_season (standings, game_log).
        team_averages: DataFrame with team GF/GA averages.
        home_ice_advantage: Home ice advantage value.

    Returns:
        A dictionary containing the winner of each round and the Stanley Cup Final.
    """
    
    standings, _ = standings_tuple
    all_winners = {}
    
    # --- Round 1: Setup and Simulation ---
    
    # 1. Setup Matchups (using Division/Wild Card rules)
    rs_results = determine_champions_and_playoffs(standings_tuple)
    playoff_teams_dict = rs_results['Playoff_Teams']
    
    r1_matchups = setup_first_round_matchups(standings_tuple, {'Playoff_Teams': playoff_teams_dict})
    all_winners['Round1_Matchups'] = r1_matchups # Optional logging
    
    # 2. Simulate Round 1
    r1_winners = simulate_playoff_round(r1_matchups, team_averages, home_ice_advantage)
    all_winners.update(r1_winners)
    
    # --- Round 2: Setup and Simulation (Division Finals) ---

    # 3. Setup Matchups (fixed bracket, home ice by RS points)
    r2_matchups = setup_later_round_matchups(r1_winners, 'Round2', standings)
    all_winners['Round2_Matchups'] = r2_matchups # Optional logging

    # 4. Simulate Round 2
    r2_winners = simulate_playoff_round(r2_matchups, team_averages, home_ice_advantage)
    all_winners.update(r2_winners)
    
    # --- Round 3: Setup and Simulation (Conference Finals) ---

    # 5. Setup Matchups (fixed bracket, home ice by RS points)
    r3_matchups = setup_later_round_matchups(r2_winners, 'Round3', standings)
    all_winners['Round3_Matchups'] = r3_matchups # Optional logging

    # 6. Simulate Round 3 (ECF, WCF)
    r3_winners = simulate_playoff_round(r3_matchups, team_averages, home_ice_advantage)
    all_winners.update(r3_winners)
    
    # --- Round 4: Setup and Simulation (Stanley Cup Final) ---
    
    # 7. Setup Matchups (fixed bracket, home ice by RS points)
    scf_matchup = setup_later_round_matchups(r3_winners, 'SCF', standings)
    #all_winners['SCF_Matchup'] = scf_matchup # Optional logging

    # 8. Simulate Round 4
    scf_winner_dict = simulate_playoff_round(scf_matchup, team_averages, home_ice_advantage)
    all_winners.update(scf_winner_dict)
    
    # 9. Return the final winner
    all_winners['Stanley_Cup_Champion'] = scf_winner_dict['SCF']
    
    return all_winners


In [22]:
def simulate_full_season(full_season_schedule: pd.DataFrame) -> pd.DataFrame:
    """
    Simulates a single full NHL season (talent draw, regular season, playoffs).
    
    The output is structured by the team's randomly assigned GF% rank (1 to 32)
    for easy aggregation across many simulation runs.
    
    Returns:
        pd.DataFrame: A 32-row DataFrame with results aggregated by GF% Rank.
    """
    
    # 1. Assign Team Talent (GF% and GF/GA)
    team_averages = assign_team_talent(gfdist_mean, gfdist_std, goals_per_game_leagueavg)
    
    # 2. Simulate Regular Season (determines standings)
    standings_tuple = simulate_regular_season(full_season_schedule, team_averages, home_ice_advantage)
    standings, _ = standings_tuple

    # 3. Determine Regular Season Awards
    rs_results = determine_champions_and_playoffs(standings_tuple)
    champions = rs_results['Champions']
    playoff_teams_by_conf = rs_results['Playoff_Teams']
    all_playoff_teams_list = playoff_teams_by_conf['Eastern'] + playoff_teams_by_conf['Western']
    all_playoff_teams = set(all_playoff_teams_list)

    # 4. Simulate Postseason (determines playoff wins)
    postseason_results = simulate_full_postseason(standings_tuple, team_averages, home_ice_advantage)
    
    # 5. Determine Playoff Round Wins for each team
    # Map team code to number of rounds won (0 to 4)
    playoff_rounds_won = {team: -1 for team in standings.index}

    for team in all_playoff_teams:
        playoff_rounds_won[team] = 0

    scf_winner = postseason_results.get('Stanley_Cup_Champion')
    #ecf_winner = postseason_results.get('ECF')
    #wcf_winner = postseason_results.get('WCF')

    # Collect winners from each round
    
    # List of teams that won Round 1 (R2 participants)
    r1_winners_codes = [v for k, v in postseason_results.items() if (k.startswith('EC_') or k.startswith('WC_')) and (k.endswith('1') 
                        or k.endswith('2')) and ('Matchups' not in k)]
    
    # Teams that won Round 2 (R3 participants)
    r2_winners_codes = [v for k, v in postseason_results.items() if k.endswith('F') and not k.endswith('CF') and 'Matchups' not in k]
    
    # Teams that won Round 3 (SCF participants, 4 rounds won total for Cup Winner)
    r3_winners_codes = [v for k, v in postseason_results.items() if k in ['ECF', 'WCF'] and 'Matchups' not in k]

    # Assign rounds won based on the final known round achieved
    
    if scf_winner:
        playoff_rounds_won[scf_winner] = 4
    
    for team in r3_winners_codes:
        if playoff_rounds_won.get(team, 0) < 3:
            playoff_rounds_won[team] = 3
        
    for team in r2_winners_codes:
        if playoff_rounds_won.get(team, 0) < 2:
            playoff_rounds_won[team] = 2
        
    for team in r1_winners_codes:
        if playoff_rounds_won.get(team, 0) < 1:
            playoff_rounds_won[team] = 1

    # --- 6. Consolidate and Rank Results by GF% ---
    
    # Merge talent, standings, and playoff data
    results_df = team_averages[['GF_percent', 'GF_percent_rank']].copy()
    
    # Add regular season stats
    results_df = results_df.join(standings[['GF', 'GA', 'GD', 'PTS']], how='inner') # Inner join ensures we only process 32 teams
    
    # Add playoff indicators
    results_df['Rounds_Won'] = results_df.index.map(playoff_rounds_won).astype(int)
    
    # Add regular season awards
    
    # Create indicator variables for regular season awards and playoff outcomes

    results_df['President_Trophy'] = (results_df.index == champions['Presidents_Trophy_Winner']).astype(int)
    results_df['Conf_Best_Record'] = results_df.index.map(
        lambda x: 1 if x == champions['Eastern_Conference_Winner'] or x == champions['Western_Conference_Winner'] else 0).astype(int)

    # Division winners
    all_division_winners = [champions['Atlantic_Division_Winner'], champions['Metropolitan_Division_Winner'], champions['Central_Division_Winner'],
                            champions['Pacific_Division_Winner']]
    results_df['Division_Winner'] = results_df.index.map(lambda x: 1 if x in all_division_winners else 0).astype(int)
    
    # Playoff outcomes
    results_df['Made_Playoffs'] = results_df.index.map(lambda x: 1 if x in all_playoff_teams else 0).astype(int)
    results_df['Won_1_Round'] = (results_df['Rounds_Won'] >= 1).astype(int)
    results_df['Won_2_Rounds'] = (results_df['Rounds_Won'] >= 2).astype(int)
    results_df['Won_3_Rounds'] = (results_df['Rounds_Won'] >= 3).astype(int)
    results_df['Won_Stanley_Cup'] = (results_df['Rounds_Won'] == 4).astype(int)
    
    # Select final output columns and ensure correct order
    final_cols = ['GF_percent', 'GF_percent_rank', 'GF', 'GA', 'GD', 'PTS', 'President_Trophy', 
                  'Conf_Best_Record', 'Division_Winner', 'Made_Playoffs',
                  'Won_1_Round', 'Won_2_Rounds', 'Won_3_Rounds', 'Won_Stanley_Cup']
    
    return results_df[final_cols]


In [36]:
num_simulations = 1
all_simulation_results = []

#full_season_schedule = pd.read_csv('nhl_schedule_2025.csv')

for i in range(num_simulations):
    if i % 1000 == 0:
        print(f"Running simulation {i+1}/{num_simulations} ...")

    season_result = simulate_full_season(full_season_schedule)
    all_simulation_results.append(season_result)

combined_results_raw = pd.concat(all_simulation_results)

# And once I've got this I can start doing analyses

Running simulation 1/1 ...


In [37]:
combined_results_raw

,GF_percent,GF_percent_rank,GF,GA,GD,PTS,President_Trophy,Conf_Best_Record,Division_Winner,Made_Playoffs,Won_1_Round,Won_2_Rounds,Won_3_Rounds,Won_Stanley_Cup
ANA,0.445185,28,245,228,17,89,0,0,0,0,0,0,0,0
BOS,0.501105,17,231,227,4,94,0,0,0,1,1,0,0,0
BUF,0.520640,8,221,222,-1,85,0,0,0,0,0,0,0,0
CGY,0.505018,14,263,219,44,91,0,0,0,1,1,0,0,0
CAR,0.528256,4,243,230,13,99,0,0,0,1,0,0,0,0
CHI,0.486503,23,236,263,-27,83,0,0,0,0,0,0,0,0
COL,0.523990,6,249,198,51,113,1,1,1,1,1,1,0,0
CBJ,0.510743,12,228,223,5,86,0,0,0,0,0,0,0,0
DAL,0.528907,3,252,245,7,91,0,0,0,1,0,0,0,0
DET,0.507809,13,217,239,-22,90,0,0,0,1,0,0,0,0


In [35]:
combined_results_raw.to_csv('likens_project_results.csv', index=False)

In [23]:
gfdist_std_likens = 0.03
gfdist_std_20222025 = 0.05927

In [25]:
num_simulations = 100000
all_simulation_results = []

#full_season_schedule = pd.read_csv('nhl_schedule_2025.csv')

for i in range(num_simulations):
    if i % 1000 == 0:
        print(f"Running simulation {i+1}/{num_simulations} ...")

    season_result = simulate_full_season_likens(full_season_schedule)
    all_simulation_results.append(season_result)

combined_results_raw = pd.concat(all_simulation_results)


Running simulation 1/100000 ...
Running simulation 1001/100000 ...
Running simulation 2001/100000 ...
Running simulation 3001/100000 ...
Running simulation 4001/100000 ...
Running simulation 5001/100000 ...
Running simulation 6001/100000 ...
Running simulation 7001/100000 ...
Running simulation 8001/100000 ...
Running simulation 9001/100000 ...
Running simulation 10001/100000 ...
Running simulation 11001/100000 ...
Running simulation 12001/100000 ...
Running simulation 13001/100000 ...
Running simulation 14001/100000 ...
Running simulation 15001/100000 ...
Running simulation 16001/100000 ...
Running simulation 17001/100000 ...
Running simulation 18001/100000 ...
Running simulation 19001/100000 ...
Running simulation 20001/100000 ...
Running simulation 21001/100000 ...
Running simulation 22001/100000 ...
Running simulation 23001/100000 ...
Running simulation 24001/100000 ...
Running simulation 25001/100000 ...
Running simulation 26001/100000 ...
Running simulation 27001/100000 ...
Runni

In [26]:
combined_results_raw.to_csv('likens_project_sens_likens.csv', index=False)

In [ ]:
# Sensitivity analysis with talent distribution SD 0.03, used in the original Likens analysis

def simulate_full_season_likens(full_season_schedule: pd.DataFrame) -> pd.DataFrame:
    """
    Simulates a single full NHL season (talent draw, regular season, playoffs).
    
    The output is structured by the team's randomly assigned GF% rank (1 to 32)
    for easy aggregation across many simulation runs.
    
    Returns:
        pd.DataFrame: A 32-row DataFrame with results aggregated by GF% Rank.
    """
    
    # 1. Assign Team Talent (GF% and GF/GA)
    team_averages = assign_team_talent(gfdist_mean, gfdist_std_likens, goals_per_game_leagueavg)
    
    # 2. Simulate Regular Season (determines standings)
    standings_tuple = simulate_regular_season(full_season_schedule, team_averages, home_ice_advantage)
    standings, _ = standings_tuple

    # 3. Determine Regular Season Awards
    rs_results = determine_champions_and_playoffs(standings_tuple)
    champions = rs_results['Champions']
    playoff_teams_by_conf = rs_results['Playoff_Teams']
    all_playoff_teams_list = playoff_teams_by_conf['Eastern'] + playoff_teams_by_conf['Western']
    all_playoff_teams = set(all_playoff_teams_list)

    # 4. Simulate Postseason (determines playoff wins)
    postseason_results = simulate_full_postseason(standings_tuple, team_averages, home_ice_advantage)
    
    # 5. Determine Playoff Round Wins for each team
    # Map team code to number of rounds won (0 to 4)
    playoff_rounds_won = {team: -1 for team in standings.index}

    for team in all_playoff_teams:
        playoff_rounds_won[team] = 0

    scf_winner = postseason_results.get('Stanley_Cup_Champion')
    #ecf_winner = postseason_results.get('ECF')
    #wcf_winner = postseason_results.get('WCF')

    # Collect winners from each round
    
    # List of teams that won Round 1 (R2 participants)
    r1_winners_codes = [v for k, v in postseason_results.items() if (k.startswith('EC_') or k.startswith('WC_')) and (k.endswith('1') 
                        or k.endswith('2')) and ('Matchups' not in k)]
    
    # Teams that won Round 2 (R3 participants)
    r2_winners_codes = [v for k, v in postseason_results.items() if k.endswith('F') and not k.endswith('CF') and 'Matchups' not in k]
    
    # Teams that won Round 3 (SCF participants, 4 rounds won total for Cup Winner)
    r3_winners_codes = [v for k, v in postseason_results.items() if k in ['ECF', 'WCF'] and 'Matchups' not in k]

    # Assign rounds won based on the final known round achieved
    
    if scf_winner:
        playoff_rounds_won[scf_winner] = 4
    
    for team in r3_winners_codes:
        if playoff_rounds_won.get(team, 0) < 3:
            playoff_rounds_won[team] = 3
        
    for team in r2_winners_codes:
        if playoff_rounds_won.get(team, 0) < 2:
            playoff_rounds_won[team] = 2
        
    for team in r1_winners_codes:
        if playoff_rounds_won.get(team, 0) < 1:
            playoff_rounds_won[team] = 1

    # --- 6. Consolidate and Rank Results by GF% ---
    
    # Merge talent, standings, and playoff data
    results_df = team_averages[['GF_percent', 'GF_percent_rank']].copy()
    
    # Add regular season stats
    results_df = results_df.join(standings[['GF', 'GA', 'GD', 'PTS']], how='inner') # Inner join ensures we only process 32 teams
    
    # Add playoff indicators
    results_df['Rounds_Won'] = results_df.index.map(playoff_rounds_won).astype(int)
    
    # Add regular season awards
    
    # Create indicator variables for regular season awards and playoff outcomes

    results_df['President_Trophy'] = (results_df.index == champions['Presidents_Trophy_Winner']).astype(int)
    results_df['Conf_Best_Record'] = results_df.index.map(
        lambda x: 1 if x == champions['Eastern_Conference_Winner'] or x == champions['Western_Conference_Winner'] else 0).astype(int)

    # Division winners
    all_division_winners = [champions['Atlantic_Division_Winner'], champions['Metropolitan_Division_Winner'], champions['Central_Division_Winner'],
                            champions['Pacific_Division_Winner']]
    results_df['Division_Winner'] = results_df.index.map(lambda x: 1 if x in all_division_winners else 0).astype(int)
    
    # Playoff outcomes
    results_df['Made_Playoffs'] = results_df.index.map(lambda x: 1 if x in all_playoff_teams else 0).astype(int)
    results_df['Won_1_Round'] = (results_df['Rounds_Won'] >= 1).astype(int)
    results_df['Won_2_Rounds'] = (results_df['Rounds_Won'] >= 2).astype(int)
    results_df['Won_3_Rounds'] = (results_df['Rounds_Won'] >= 3).astype(int)
    results_df['Won_Stanley_Cup'] = (results_df['Rounds_Won'] == 4).astype(int)
    
    # Select final output columns and ensure correct order
    final_cols = ['GF_percent', 'GF_percent_rank', 'GF', 'GA', 'GD', 'PTS', 'President_Trophy', 
                  'Conf_Best_Record', 'Division_Winner', 'Made_Playoffs',
                  'Won_1_Round', 'Won_2_Rounds', 'Won_3_Rounds', 'Won_Stanley_Cup']
    
    return results_df[final_cols]


In [27]:
# Sensitivity analysis with talent distribution SD 0.05927 from 4 32-team seasons

def simulate_full_season_20222025(full_season_schedule: pd.DataFrame) -> pd.DataFrame:
    """
    Simulates a single full NHL season (talent draw, regular season, playoffs).
    
    The output is structured by the team's randomly assigned GF% rank (1 to 32)
    for easy aggregation across many simulation runs.
    
    Returns:
        pd.DataFrame: A 32-row DataFrame with results aggregated by GF% Rank.
    """
    
    # 1. Assign Team Talent (GF% and GF/GA)
    team_averages = assign_team_talent(gfdist_mean, gfdist_std_20222025, goals_per_game_leagueavg)
    
    # 2. Simulate Regular Season (determines standings)
    standings_tuple = simulate_regular_season(full_season_schedule, team_averages, home_ice_advantage)
    standings, _ = standings_tuple

    # 3. Determine Regular Season Awards
    rs_results = determine_champions_and_playoffs(standings_tuple)
    champions = rs_results['Champions']
    playoff_teams_by_conf = rs_results['Playoff_Teams']
    all_playoff_teams_list = playoff_teams_by_conf['Eastern'] + playoff_teams_by_conf['Western']
    all_playoff_teams = set(all_playoff_teams_list)

    # 4. Simulate Postseason (determines playoff wins)
    postseason_results = simulate_full_postseason(standings_tuple, team_averages, home_ice_advantage)
    
    # 5. Determine Playoff Round Wins for each team
    # Map team code to number of rounds won (0 to 4)
    playoff_rounds_won = {team: -1 for team in standings.index}

    for team in all_playoff_teams:
        playoff_rounds_won[team] = 0

    scf_winner = postseason_results.get('Stanley_Cup_Champion')
    #ecf_winner = postseason_results.get('ECF')
    #wcf_winner = postseason_results.get('WCF')

    # Collect winners from each round
    
    # List of teams that won Round 1 (R2 participants)
    r1_winners_codes = [v for k, v in postseason_results.items() if (k.startswith('EC_') or k.startswith('WC_')) and (k.endswith('1') 
                        or k.endswith('2')) and ('Matchups' not in k)]
    
    # Teams that won Round 2 (R3 participants)
    r2_winners_codes = [v for k, v in postseason_results.items() if k.endswith('F') and not k.endswith('CF') and 'Matchups' not in k]
    
    # Teams that won Round 3 (SCF participants, 4 rounds won total for Cup Winner)
    r3_winners_codes = [v for k, v in postseason_results.items() if k in ['ECF', 'WCF'] and 'Matchups' not in k]

    # Assign rounds won based on the final known round achieved
    
    if scf_winner:
        playoff_rounds_won[scf_winner] = 4
    
    for team in r3_winners_codes:
        if playoff_rounds_won.get(team, 0) < 3:
            playoff_rounds_won[team] = 3
        
    for team in r2_winners_codes:
        if playoff_rounds_won.get(team, 0) < 2:
            playoff_rounds_won[team] = 2
        
    for team in r1_winners_codes:
        if playoff_rounds_won.get(team, 0) < 1:
            playoff_rounds_won[team] = 1

    # --- 6. Consolidate and Rank Results by GF% ---
    
    # Merge talent, standings, and playoff data
    results_df = team_averages[['GF_percent', 'GF_percent_rank']].copy()
    
    # Add regular season stats
    results_df = results_df.join(standings[['GF', 'GA', 'GD', 'PTS']], how='inner') # Inner join ensures we only process 32 teams
    
    # Add playoff indicators
    results_df['Rounds_Won'] = results_df.index.map(playoff_rounds_won).astype(int)
    
    # Add regular season awards
    
    # Create indicator variables for regular season awards and playoff outcomes

    results_df['President_Trophy'] = (results_df.index == champions['Presidents_Trophy_Winner']).astype(int)
    results_df['Conf_Best_Record'] = results_df.index.map(
        lambda x: 1 if x == champions['Eastern_Conference_Winner'] or x == champions['Western_Conference_Winner'] else 0).astype(int)

    # Division winners
    all_division_winners = [champions['Atlantic_Division_Winner'], champions['Metropolitan_Division_Winner'], champions['Central_Division_Winner'],
                            champions['Pacific_Division_Winner']]
    results_df['Division_Winner'] = results_df.index.map(lambda x: 1 if x in all_division_winners else 0).astype(int)
    
    # Playoff outcomes
    results_df['Made_Playoffs'] = results_df.index.map(lambda x: 1 if x in all_playoff_teams else 0).astype(int)
    results_df['Won_1_Round'] = (results_df['Rounds_Won'] >= 1).astype(int)
    results_df['Won_2_Rounds'] = (results_df['Rounds_Won'] >= 2).astype(int)
    results_df['Won_3_Rounds'] = (results_df['Rounds_Won'] >= 3).astype(int)
    results_df['Won_Stanley_Cup'] = (results_df['Rounds_Won'] == 4).astype(int)
    
    # Select final output columns and ensure correct order
    final_cols = ['GF_percent', 'GF_percent_rank', 'GF', 'GA', 'GD', 'PTS', 'President_Trophy', 
                  'Conf_Best_Record', 'Division_Winner', 'Made_Playoffs',
                  'Won_1_Round', 'Won_2_Rounds', 'Won_3_Rounds', 'Won_Stanley_Cup']
    
    return results_df[final_cols]


In [28]:
num_simulations = 100000
all_simulation_results = []

#full_season_schedule = pd.read_csv('nhl_schedule_2025.csv')

for i in range(num_simulations):
    if i % 1000 == 0:
        print(f"Running simulation {i+1}/{num_simulations} ...")

    season_result = simulate_full_season_20222025(full_season_schedule)
    all_simulation_results.append(season_result)

combined_results_raw = pd.concat(all_simulation_results)

Running simulation 1/100000 ...
Running simulation 1001/100000 ...
Running simulation 2001/100000 ...
Running simulation 3001/100000 ...
Running simulation 4001/100000 ...
Running simulation 5001/100000 ...
Running simulation 6001/100000 ...
Running simulation 7001/100000 ...
Running simulation 8001/100000 ...
Running simulation 9001/100000 ...
Running simulation 10001/100000 ...
Running simulation 11001/100000 ...
Running simulation 12001/100000 ...
Running simulation 13001/100000 ...
Running simulation 14001/100000 ...
Running simulation 15001/100000 ...
Running simulation 16001/100000 ...
Running simulation 17001/100000 ...
Running simulation 18001/100000 ...
Running simulation 19001/100000 ...
Running simulation 20001/100000 ...
Running simulation 21001/100000 ...
Running simulation 22001/100000 ...
Running simulation 23001/100000 ...
Running simulation 24001/100000 ...
Running simulation 25001/100000 ...
Running simulation 26001/100000 ...
Running simulation 27001/100000 ...
Runni

In [29]:
combined_results_raw.to_csv('likens_project_sens_20222025.csv', index=False)